# Marlong · 趋势策略回测演示

策略逻辑：MA5/20 金叉 + MACD 多头确认 + SAR 过滤

适用：A 股日线趋势交易

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

from config.settings import DEFAULT_CONFIG
from data.loader import load_data
from strategy.trend_ma import TrendMAStrategy
from backtest.engine import BacktestEngine
from backtest.metrics import print_metrics
import matplotlib.pyplot as plt
import pandas as pd

print('模块加载成功')

In [ ]:
# 修改配置
cfg = DEFAULT_CONFIG.copy()
cfg['symbol'] = '300274'
cfg['symbol_name'] = '阳光电源'
cfg['start_date'] = '20230101'
cfg['end_date'] = '20260101'
cfg['initial_capital'] = 100_000
cfg['stop_loss_pct'] = 0.07
cfg['take_profit_pct'] = 0.20

print(f"回测目标：{cfg['symbol_name']}（{cfg['symbol']}）")
print(f"时间范围：{cfg['start_date']} → {cfg['end_date']}")

In [ ]:
# 加载数据（首次自动拉取，后续读缓存）
df = load_data(
    symbol=cfg['symbol'],
    start_date=cfg['start_date'],
    end_date=cfg['end_date'],
    data_dir=cfg['data_dir'],
)
print(f'加载数据：{len(df)} 条')
df.tail(3)

In [ ]:
# 策略运行 & 信号生成
strategy = TrendMAStrategy(cfg)
df = strategy.run(df)

buy_signals = (df['signal'] == 1).sum()
sell_signals = (df['signal'] == -1).sum()
print(f'买入信号：{buy_signals} 次，卖出信号：{sell_signals} 次')

In [ ]:
# 回测执行
engine = BacktestEngine(cfg)
result = engine.run(df)
print_metrics(result['metrics'])

In [ ]:
# 资产曲线
equity = result['equity']
plt.figure(figsize=(13, 4))
plt.plot(equity.index, equity.values, color='#2980b9', linewidth=1.5)
plt.axhline(cfg['initial_capital'], color='gray', linewidth=0.8, linestyle='--')
plt.fill_between(equity.index, cfg['initial_capital'], equity.values,
                 where=(equity.values >= cfg['initial_capital']), alpha=0.15, color='#27ae60')
plt.fill_between(equity.index, cfg['initial_capital'], equity.values,
                 where=(equity.values < cfg['initial_capital']), alpha=0.15, color='#e74c3c')
plt.title(f"Marlong · {cfg['symbol_name']} 资产曲线")
plt.ylabel('总资产（元）')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 查看交易记录
trades = result['trades']
completed = trades[trades['pnl'].notna()].copy()
print(f'完成交易：{len(completed)} 笔')
completed[['entry_date','entry_price','exit_date','exit_price','shares','pnl','exit_reason']]